# updt.pro CV MVP — Colab GPU run

Runs the perception + assisted-tagging pipeline on a Colab **GPU** (escapes the local M1 16 GB memory wall). Produces an event log (possession / passes / carries / zone entries) with confidence scores.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is enough).

Set `REPO_URL` below to your GitHub repo, then `Runtime → Run all`.

In [ ]:
# 0) Confirm a GPU is attached
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime>GPU')

In [ ]:
# 1) Clone the repo  (edit REPO_URL + REPO_DIR to match your GitHub repo)
REPO_URL = 'https://github.com/<your-username>/updtsupervision.git'
REPO_DIR = 'updtsupervision'
import os
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR
!git log --oneline -1

In [ ]:
# 2) Install deps. Colab's CUDA torch is reused (requirements pin torch>=2.2).
#    NOTE: requirements pin numpy<2 — if Colab downgrades numpy, it may ask you to
#    'Restart runtime'. Do so, then re-run from cell 1 (skips the clone).
!pip install -q -r requirements.txt

In [ ]:
# 3) Download pretrained weights + one DFL sample clip (no Roboflow API key needed)
!python scripts/download_assets.py
!ls -lh data/models data/raw_matches

In [ ]:
# 4) Run the pipeline on GPU. 500 frames (~20s) is enough to surface real events;
#    drop --max-frames for the full clip (fast on a T4). First run downloads SigLIP (~1.5GB).
!PYTORCH_ENABLE_MPS_FALLBACK=1 python run.py \
    --source data/raw_matches/2e57b9_0.mp4 \
    --device cuda --max-frames 500 \
    --output-dir outputs/colab

In [ ]:
# 5) Inspect the run summary + the event log the frontend/DB will consume
import json, pandas as pd
print(json.dumps(json.load(open('outputs/colab/run_summary.json')), indent=2))
events = json.load(open('outputs/colab/events.json'))
df = pd.DataFrame(events)
print(f'\n{len(df)} events')
if len(df):
    print(df['type'].value_counts().to_dict())
    cols = ['type','team_id','start_ts','end_ts','length_m','confidence','review_status']
    display(df[[c for c in cols if c in df.columns]].head(25))

In [ ]:
# 6) Visual sanity check: detections + tracks drawn on a few frames
import cv2, supervision as sv, matplotlib.pyplot as plt
from ultralytics import YOLO
model = YOLO('data/models/players.pt')
cap = cv2.VideoCapture('data/raw_matches/2e57b9_0.mp4')
box, lab = sv.BoxAnnotator(), sv.LabelAnnotator()
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, fno in zip(axes.ravel(), [100, 200, 300, 400]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fno)
    ok, img = cap.read()
    if not ok:
        continue
    det = sv.Detections.from_ultralytics(model.predict(img, conf=0.3, device=0, verbose=False)[0])
    ann = lab.annotate(box.annotate(img.copy(), det), det)
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)); ax.set_title(f'frame {fno}'); ax.axis('off')
cap.release(); plt.tight_layout(); plt.show()